[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/08_streaming.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-仓库主页-181717?logo=github)](https://github.com/Luca5702/langgraph-study)

**📚 课程导航：** [01](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/01_hello_graph.ipynb) · [02](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/02_state_reducers.ipynb) · [03](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb) · [04](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/04_agent_tools.ipynb) · [05](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/05_persistence_memory.ipynb) · [06](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/06_human_in_the_loop.ipynb) · [07](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb) · **[08](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/08_streaming.ipynb)** · [09](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/09_subgraphs.ipynb) · [10](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/10_functional_api.ipynb) · [11](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/11_production.ipynb)

[⬅️ 上一课：第 7 课：时间旅行（Replay & Fork）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb) ｜ [➡️ 下一课：第 9 课：子图与多智能体 Handoff](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/09_subgraphs.ipynb)

---

# 第 8 课：流式输出 —— stream_mode、v2 统一格式、stream_events(v3)

LangGraph 有两层流式 API：
1) graph.stream(..., stream_mode=..., version="v2")   —— 原始事件
  values   : 每个超步后的完整 state
  updates  : 每个节点返回的增量
  messages : LLM 的 token（(chunk, metadata)）
  custom   : 节点里用 get_stream_writer() 推送的任意数据（进度条等）
  checkpoints / tasks / debug : 调试用
   version="v2" 让每个 chunk 都是 {"type", "ns", "data"}，格式统一（推荐）。
2) graph.stream_events(..., version="v3")  —— 新的"类型化投影"
  stream.messages / stream.values / stream.subgraphs / stream.output ...

In [1]:
#@title 🔧 第 0 步：初始化环境（在 Colab 中点击左侧 ▶ 即可自动安装依赖，本地运行自动跳过） { display-mode: "form" }
import importlib.util, subprocess, sys
if importlib.util.find_spec('langgraph') is None:
    print('⏳ 正在安装 LangGraph 环境（约 10 秒）...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'langgraph', 'langchain-core', 'langgraph-checkpoint-sqlite'])
    print('✅ 安装完成！')
else:
    import importlib.metadata as m
    print(f'✅ 环境已就绪：langgraph={m.version("langgraph")}')
from pathlib import Path
import urllib.request
for p in [Path('lessons'), Path.cwd().parent / 'lessons']:
    if (p / 'fake_llm.py').exists():
        sys.path.insert(0, str(p.resolve()))
        break
else:
    if not Path('fake_llm.py').exists():
        urllib.request.urlretrieve('https://raw.githubusercontent.com/Luca5702/langgraph-study/main/lessons/fake_llm.py', 'fake_llm.py')


✅ 环境已就绪：langgraph=1.2.22


In [2]:
import warnings

from langgraph.config import get_stream_writer
from langgraph.graph import START, MessagesState, StateGraph

from fake_llm import get_model

# v3 目前是 experimental（beta）；需在 import langchain_core 之后设置过滤器
warnings.filterwarnings("ignore", message=".*v3 streaming protocol.*")

model = get_model()


def prepare(state: MessagesState):
    writer = get_stream_writer()
    for p in (30, 60, 100):
        writer({"progress": p, "msg": "正在检索资料..."})  # -> custom 流
    return {}


def answer(state: MessagesState):
    return {"messages": [model.invoke(state["messages"])]}


graph = (
    StateGraph(MessagesState)
    .add_node(prepare).add_node(answer)
    .add_edge(START, "prepare").add_edge("prepare", "answer")
    .compile()
)
inputs = {"messages": [{"role": "user", "content": "介绍一下 LangGraph"}]}

## ▶️ 运行实验

In [3]:
print("== 1. stream_mode='updates'（v1 默认格式）==")
for chunk in graph.stream(inputs, stream_mode="updates"):
    print("  ", {k: (type(v).__name__ if v else v) for k, v in chunk.items()})

print("\n== 2. 多模式 + version='v2' ==")
for part in graph.stream(inputs, stream_mode=["updates", "messages", "custom"], version="v2"):
    if part["type"] == "custom":
        print(f"   [custom] {part['data']}")
    elif part["type"] == "updates":
        print(f"   [updates] 节点 {list(part['data'])} 完成")
    elif part["type"] == "messages":
        chunk, meta = part["data"]
        print(chunk.content, end="|", flush=True)  # 用 | 分隔，看清逐 token
print()

print("\n== 3. stream_events(version='v3')：类型化投影 ==")
stream = graph.stream_events(inputs, version="v3")
for message in stream.messages:
    print(f"   来自节点 {message.node}: ", end="")
    for token in message.text:
        print(token, end="", flush=True)
    print()
print("   最终 state 消息数:", len(stream.output["messages"]))

== 1. stream_mode='updates'（v1 默认格式）==
   {'prepare': None}
   {'answer': 'dict'}

== 2. 多模式 + version='v2' ==
   [custom] {'progress': 30, 'msg': '正在检索资料...'}
   [custom] {'progress': 60, 'msg': '正在检索资料...'}
   [custom] {'progress': 100, 'msg': '正在检索资料...'}
   [updates] 节点 ['prepare'] 完成
（|

假|

模|

型|

）|

我|

... (65 outputs left out)

题

吧

！


   最终 state 消息数: 2


## ✏️ 练习：
 1. 用 stream_mode="values" 打印每个超步后的消息条数。
 2. 结合第 6 课：stream_events 后检查 stream.interrupted / stream.interrupts。
 3. 阅读 docs/src/oss/langgraph/event-streaming.mdx 的 "Build your own projection"。

In [4]:
# 在这里写你的练习代码 👇


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/08_streaming.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-仓库主页-181717?logo=github)](https://github.com/Luca5702/langgraph-study)

**📚 课程导航：** [01](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/01_hello_graph.ipynb) · [02](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/02_state_reducers.ipynb) · [03](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb) · [04](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/04_agent_tools.ipynb) · [05](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/05_persistence_memory.ipynb) · [06](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/06_human_in_the_loop.ipynb) · [07](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb) · **[08](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/08_streaming.ipynb)** · [09](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/09_subgraphs.ipynb) · [10](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/10_functional_api.ipynb) · [11](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/11_production.ipynb)

[⬅️ 上一课：第 7 课：时间旅行（Replay & Fork）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb) ｜ [➡️ 下一课：第 9 课：子图与多智能体 Handoff](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/09_subgraphs.ipynb)

---